Fingerprinting

This workbook uses a IDL_python bridge to run fingerprinting analysis on seasonal runoff data across New Zealand. It defines functions which read previously created runoff datafiles and setup the data_obs, data_scen, and noise vectors. These vectors are then input into Daithi Stone's gendetec IDL code which returns the regression coefficients for the multiple linear regression problem as follows: data_obs = beta_est * data_scen + residual. 

Results are saved in ~/figures/fingerprinting/beta_est/ if using a single truncation and ""/scan_beta_est/ if depicting a range of truncation options. 

In [3]:
import numpy as np
import pandas as pd
import xarray as xr
from functools import reduce
import matplotlib.pyplot as plt
from idlpy import *
from scipy import stats
import statsmodels.api as sm


import seaborn as sns
import cartopy.crs as ccrs

In [7]:
def get_obs_data(season, mem, resample_interval):
    
    if season == 'all_concatenated':
        
        df_list = []
        for season in ['winter','spring','summer','autumn']:
            df = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_all_'+season+'_ens'+mem+'.csv')
            df['time'] = pd.to_datetime(df['time'])
            df = df.set_index('time')
            df = df['1979':'2019']
            
            if resample_interval != 'annual':
                df = df.resample(resample_interval).mean()

            df_list.append(df)
            
        all_seasons = pd.concat(df_list)
        all_seasons.sort_index(inplace=True)
        return_array = all_seasons.values.flatten()
            
        
    else:
        df = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_all_'+season+'_ens'+mem+'.csv')
        df['time'] = pd.to_datetime(df['time'])
        df = df.set_index('time')
        df = df['1979':'2019']
        
        if resample_interval != 'annual':
            df = df.resample(resample_interval).mean()
        
        return_array = df.values.flatten()
        
        del df

    return(return_array)

In [8]:
def get_mean_and_noise(season,scen, mem, resample_interval):
    
    if season == 'all':
        ens_mean_season_list = []
        for season in ['winter','spring','summer','autumn']:
            ens_mean = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_'+scen+'_'+season+'_ensMean.csv')
            ens_mean.time = pd.to_datetime(ens_mean.time)
            ens_mean.set_index('time',inplace=True)
            
            if resample_interval != 'annual':
                ens_mean = ens_mean.resample(resample_interval).mean()
            
            ens_mean_season_list.append(ens_mean)
            
        ens_mean_all_seasons = pd.concat(ens_mean_season_list)
        ens_mean_all_seasons.sort_index(inplace=True)
        ens_mean = ens_mean_all_seasons
        
        noise = []
        for ens_mem in range(1,51):
            ens_mem_season_list = []
            for season in ['winter','spring','summer','autumn']:
                number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)
                mem_df = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_'+scen+'_'+season+'_ens'+number+'.csv')
                mem_df.time = pd.to_datetime(mem_df.time)
                mem_df.set_index('time',inplace=True)
                
                if resample_interval != 'annual':
                    mem_df = mem_df.resample(resample_interval).mean()
                
                ens_mem_season_list.append(mem_df)

            ens_mem_all_seasons = pd.concat(ens_mem_season_list)
            ens_mem_all_seasons.sort_index(inplace = True)
            
            
            diff = ens_mem_all_seasons - ens_mean_all_seasons

            noise.append(diff.values.flatten())

    
    else:
        
        ens_mean = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_'+scen+'_'+season+'_ensMean.csv')
        ens_mean.time = pd.to_datetime(ens_mean.time)
        ens_mean.set_index('time',inplace=True)
        
        if resample_interval != 'annual':
            ens_mean = ens_mean.resample(resample_interval).mean()

        noise = []
        for ens_mem in range(1,51):
            number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)

            if number == mem:
                continue

            mem_df = pd.read_csv('/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/gaussian/runoff/ECHAM5_'+scen+'_'+season+'_ens'+number+'.csv')
            mem_df.time = pd.to_datetime(mem_df.time)
            mem_df.set_index('time',inplace=True)
            
            if resample_interval != 'annual':
                mem_df = mem_df.resample(resample_interval).mean()

            diff = mem_df - ens_mean
            
            del mem_df

            noise.append(diff.values.flatten())

    ens_mean_array = ens_mean.values.flatten()
    
    del ens_mean
    
    return((ens_mean_array, np.array(noise)))



In [ ]:
'''
Perfect model testing for EACH SEASON INDIVIDUALLY, using each ensemble member as the observations.
'''


seasons = ['summer','autumn','winter','spring']

for resample_interval in ['annual','5Y','10Y']:

    for ens_mem in range(1,51):

        number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)

        fig,axs = plt.subplots(2,2, figsize=(12,10))
        axs = axs.flatten()

        for i,s in enumerate(seasons):    

            IDL.retall

            all_ens_mean,noise_all = get_mean_and_noise(s,'all',number,resample_interval)
            nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',number,resample_interval)

            data_obs = get_obs_data(s, number,resample_interval)

            data_scen = np.array([all_ens_mean,nat_ens_mean])

            data_noise_1 = np.concatenate((noise_all[:24],noise_nat[:24]))
            data_noise_2 = np.concatenate((noise_all[24:],noise_nat[24:]))


            trunc = np.array([i for i in range(3,49)])

            IDL.data_obs = data_obs
            IDL.data_scen = data_scen
            IDL.data_noise_1 = data_noise_1
            IDL.data_noise_2 = data_noise_2
            IDL.trunc = trunc
            IDL.scan_beta_est = []
            IDL.scan_p_resid = []
            IDL.scan_resid_sumsq = []
            IDL.p_resid = []
            IDL.resid_sumsq = []
            IDL.noise_singval = []
            IDL.z_best = []

            IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est, scan_resid_sumsq=scan_resid_sumsq, \
                    data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc, scan_beta_est=scan_beta_est,\
                    scan_p_resid=scan_p_resid,noise_singval=noise_singval,z_best=z_best")

            beta_est = IDL.beta_est
            scan_beta_est = IDL.scan_beta_est
            scan_p_resid = IDL.scan_p_resid
            scan_resid_sumsq = IDL.scan_resid_sumsq
            noise_singval = IDL.noise_singval
            z_best = IDL.z_best

            for j in range(len(scan_beta_est)):

                beta_est = scan_beta_est[j]

                best_1,low_1,up_1 = beta_est[0][0],beta_est[1][0],beta_est[2][0]
                best_2,low_2,up_2 = beta_est[0][1],beta_est[1][1],beta_est[2][1]

                axs[i].scatter([j+3]*3,[low_2,best_2,up_2],c='blue',alpha=0.7,label='nat' if j == 0 else '')
                axs[i].scatter([j+3]*3,[low_1,best_1,up_1],c='red',label='ant' if j == 0 else '')

                axs[i].vlines(j+3,low_2,up_2,colors=['blue'])
                axs[i].vlines(j+3,low_1,up_1,colors=['red'])

                axs[i].axhline(0,-0.5,1.5)
                axs[i].axhline(1,-0.5,1.5)


            axs[i].set_title(s)

            axs[i].legend()

        plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/streamflow/seasonal/'+resample_interval+'/' + 'PM_ens_' + number + '.png')

        plt.close(fig)




!null = 0
         ^
% Program code area full.


In [19]:
'''
Perfect model testing for seasons concatenated, using each ensemble member as the observations.
'''

for ens_mem in range(1,51):
    
    if ens_mem > 40:
        continue
        
    number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)

        
    fig,ax = plt.subplots(figsize=(8,6))#, sharex=True,sharey=True)

    IDL.retall

    all_ens_mean,noise_all = get_mean_and_noise('all','all',number)
    nat_ens_mean,noise_nat = get_mean_and_noise('all','nat',number)

    data_obs = get_obs_data('all', number)

    data_scen = np.array([all_ens_mean,nat_ens_mean])

    data_noise_1 = np.concatenate((noise_all[:24],noise_nat[:24]))
    data_noise_2 = np.concatenate((noise_all[24:],noise_nat[24:]))


    trunc = np.array([i for i in range(3,48)])

    IDL.data_obs = data_obs
    IDL.data_scen = data_scen
    IDL.data_noise_1 = data_noise_1
    IDL.data_noise_2 = data_noise_2
    IDL.trunc = trunc
    IDL.scan_beta_est = []
    IDL.scan_p_resid = []
    IDL.scan_resid_sumsq = []
    IDL.p_resid = []
    IDL.resid_sumsq = []
    IDL.noise_singval = []
    IDL.z_best = []

    IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est, scan_resid_sumsq=scan_resid_sumsq, \
            data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc, scan_beta_est=scan_beta_est,\
            scan_p_resid=scan_p_resid,noise_singval=noise_singval,z_best=z_best")

    beta_est = IDL.beta_est
    scan_beta_est = IDL.scan_beta_est
    scan_p_resid = IDL.scan_p_resid
    scan_resid_sumsq = IDL.scan_resid_sumsq
    noise_singval = IDL.noise_singval
    z_best = IDL.z_best

    est_obs = z_best[2]

    for j in range(len(scan_beta_est)):

        beta_est = scan_beta_est[j]

        best_1,low_1,up_1 = beta_est[0][0],beta_est[1][0],beta_est[2][0]
        best_2,low_2,up_2 = beta_est[0][1],beta_est[1][1],beta_est[2][1]

        ax.scatter([j]*3,[low_1,best_1,up_1],c='blue',label='ant' if j == 0 else '')
        ax.scatter([j]*3,[low_2,best_2,up_2],c='red',alpha=0.7,label='nat' if j == 0 else '')

        ax.vlines(j,low_1,up_1,colors=['blue'])
        ax.vlines(j,low_2,up_2,colors=['red'])

        ax.axhline(0,-0.5,1.5)
        ax.axhline(1,-0.5,1.5)


    ax.set_title(s + ', Trunc = ' + str(IDL.trunc))

    ax.legend()
        
    plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/streamflow/all_seasons/' + 'PM_ens_' + number + '.png')
    
    plt.close(fig)


In [9]:
'''
Ensemble summary at one truncation level
'''

seasons = ['summer','autumn','winter','spring']

for resample_interval in ['annual']:#'5Y','10Y','annual']:

    season_dfs = []
    
    for s in seasons:

        ensemble_members = []
        ant_uppers = []
        ant_lowers = []
        ant_bests = []

        nat_uppers = []
        nat_lowers = []
        nat_bests = []
        for ens_mem in range(1,51):

            number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)
            ensemble_members.append(number)

            IDL.retall

            all_ens_mean,noise_all = get_mean_and_noise(s,'all',number,resample_interval)
            nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',number,resample_interval)

            data_obs = get_obs_data(s, number, resample_interval)

            data_scen = np.array([all_ens_mean,nat_ens_mean])

            data_noise_1 = np.concatenate((noise_all[:24],noise_nat[:24]))
            data_noise_2 = np.concatenate((noise_all[25:],noise_nat[25:]))


            trunc = 36

            IDL.data_obs = data_obs
            IDL.data_scen = data_scen
            IDL.data_noise_1 = data_noise_1
            IDL.data_noise_2 = data_noise_2
            IDL.trunc = trunc


            IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est,\
                    data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc")

            beta_est = IDL.beta_est

            best_ant,low_ant,up_ant = beta_est[0][0],beta_est[1][0],beta_est[2][0]
            best_nat,low_nat,up_nat = beta_est[0][1],beta_est[1][1],beta_est[2][1]

            ant_uppers.append(up_ant)
            ant_lowers.append(low_ant)
            ant_bests.append(best_ant)

            nat_uppers.append(up_nat)
            nat_lowers.append(low_nat)
            nat_bests.append(best_nat)        

        df = pd.DataFrame({'ensemble_member':ensemble_members,
                          'ant_uppers':ant_uppers,
                          'ant_lowers':ant_lowers,
                          'ant_bests':ant_bests,
                          'nat_uppers':nat_uppers,
                          'nat_lowers':nat_lowers,
                          'nat_bests':nat_bests})

        season_dfs.append(df)
        
    for sort_type in ['nat_bests','ant_bests']:
        
        fig,axs = plt.subplots(2,2, figsize=(12,10))
        axs = axs.flatten()
        
        for i,s in enumerate(seasons):
            
            sorted_df = season_dfs[i].sort_values(by=sort_type)
            
            axs[i].scatter([i for i in range(len(sorted_df.nat_uppers))],sorted_df.nat_uppers,c='blue',label='nat')
            axs[i].scatter([i for i in range(len(sorted_df.nat_lowers))],sorted_df.nat_lowers,c='blue',label='')
            axs[i].scatter([i for i in range(len(sorted_df.nat_bests))],sorted_df.nat_bests,c='blue',label='')

            axs[i].scatter([i for i in range(len(sorted_df.ant_uppers))],sorted_df.ant_uppers,c='red',label='ant')
            axs[i].scatter([i for i in range(len(sorted_df.ant_lowers))],sorted_df.ant_lowers,c='red',label='')
            axs[i].scatter([i for i in range(len(sorted_df.ant_bests))],sorted_df.ant_bests,c='red',label='')

            axs[i].vlines([i for i in range(len(sorted_df.nat_lowers))],sorted_df.nat_lowers,sorted_df.nat_uppers,colors=['blue'],label='')
            axs[i].vlines([i for i in range(len(sorted_df.ant_lowers))],sorted_df.ant_lowers,sorted_df.ant_uppers,colors=['red'],label='')

            axs[i].axhline(0,-0.5,1.5)
            axs[i].axhline(1,-0.5,1.5)

            axs[i].set_title(s + ', trunc = ' + str(trunc))

            axs[i].legend()

        plt.savefig('/nesi/project/niwa00015/queenle/plots/perfect_model/ECHAM5/streamflow/seasonal/' + resample_interval + '/' + 'PM_ensemble_summary_'+sort_type+'_sort_TEST.png')

        plt.close(fig)


In [10]:
'''
Write ensemble summary csv files
'''

seasons = ['summer','autumn','winter','spring']

for resample_interval in ['annual','10Y','5Y']:

    ensemble_member = []
    ant_upper = []
    ant_lower = []
    ant_best = []
    nat_upper = []
    nat_lower = []
    nat_best = []
    season = []
    for ens_mem in range(1,51):

        number = '0' + str(ens_mem) if ens_mem < 10 else str(ens_mem)

        for i,s in enumerate(seasons):    

            IDL.retall

            all_ens_mean,noise_all = get_mean_and_noise(s,'all',number,resample_interval)
            nat_ens_mean,noise_nat = get_mean_and_noise(s,'nat',number,resample_interval)

            data_obs = get_obs_data(s, number, resample_interval)

            data_scen = np.array([all_ens_mean,nat_ens_mean])

            #data_noise_1 = np.concatenate((noise_all[:24],noise_nat[:25]))
            #data_noise_2 = np.concatenate((noise_all[24:],noise_nat[25:]))
            
            data_noise_1 = np.concatenate((noise_all[:24],noise_nat[:25]))
            data_noise_2 = np.concatenate((noise_all[24:],noise_nat[25:]))
            
            trunc = 36

            IDL.data_obs = data_obs
            IDL.data_scen = data_scen
            IDL.data_noise_1 = data_noise_1
            IDL.data_noise_2 = data_noise_2
            IDL.trunc = trunc

            IDL.run("gendetec, data_obs, data_scen, data_noise_1, beta_est, \
                    data_noise_2=data_noise_2,transform=['1*0','1*0+1*1'], trunc=trunc")

            beta_est = IDL.beta_est

            ensemble_member.append(number)
            ant_upper.append(beta_est[2][0])
            ant_lower.append(beta_est[1][0])
            ant_best.append(beta_est[0][0])
            nat_upper.append(beta_est[2][1])
            nat_lower.append(beta_est[1][1])
            nat_best.append(beta_est[0][1])
            season.append(s)
        
    df = pd.DataFrame({'ensemble_member':ensemble_member,
                      'season':season,
                      'ant_upper':ant_upper,
                      'ant_lower':ant_lower,
                      'ant_best':ant_best,
                      'nat_upper':nat_upper,
                      'nat_lower':nat_lower,
                      'nat_best':nat_best})

    df.to_csv("/nesi/project/niwa00015/queenle/data/hydro_fingerprinting/model/ECHAM5/summary_csv/sro_" +resample_interval + "_trunc_" + str(trunc) + "_original_noise.csv")
    
    del df
